# HookLink2iOS — Ocarina of Time (Switch 2 Edition)

Este notebook reaproveita a estrutura Colab do Link2iOS, mas faz a preparação do jogo no Linux usando o packer NUP do nxnative/HookLink. Depois envia NUP + RomFS + overlay ao Google Drive e grava um handoff na branch isolada `HookLink2iOS`.

**Alvo fixo:** Title ID `0400486024DEC000`, update `v131072`, GPU preset inicial `g18p`.

**Entrada:** um arquivo NUP próprio ou um arquivo compactado que contenha exatamente um NUP. RAR, ZIP, 7z e formatos tar suportados pelo bsdtar são aceitos.

O notebook não traz URL de jogo pré-configurado. Escolha um dump que você tenha direito de utilizar, enviado ao Drive ou hospedado por você.

In [ ]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq libarchive-tools
!python -m pip install --quiet requests google-api-python-client google-auth backports.zstd
!rm -rf /content/HookLink2iOS-repo
!git clone --depth 1 --branch HookLink2iOS https://github.com/joao9vegeto-stack/KintTany.git /content/HookLink2iOS-repo

from pathlib import Path
import sys
sys.path.insert(0, "/content/HookLink2iOS-repo/colab")
from hooklink2ios_prepare import prepare

print("HookLink2iOS checkout pronto.")


## 1. Conectar Google Drive

A pasta pai precisa estar compartilhada com a conta de serviço configurada em `GOOGLE_SERVICE_ACCOUNT_JSON` no repositório GitHub, para que o Actions consiga baixar os arquivos e enviar a IPA.


In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive"]
)
drive = build("drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False)
DRIVE_PARENT_FOLDER_ID = input("ID da pasta pai no Google Drive: ").strip()
assert DRIVE_PARENT_FOLDER_ID, "Informe o ID da pasta pai."
drive_parent = drive.files().get(
    fileId=DRIVE_PARENT_FOLDER_ID,
    fields="id,name,mimeType",
    supportsAllDrives=True
).execute()
assert drive_parent.get("mimeType") == "application/vnd.google-apps.folder"
print("Pasta selecionada:", drive_parent["name"])


## 2. Fornecer o dump

Escolha um NUP já no Drive ou um URL HTTPS direto seu. Se for RAR/ZIP/7z, o Colab extrai para o disco temporário, valida que existe exatamente um NUP, gera o RomFS e só então apaga o arquivo compactado.


In [ ]:
from getpass import getpass

SOURCE_MODE = input("Origem [drive/url]: ").strip().lower()
SOURCE_DRIVE_FILE_ID = ""
SOURCE_URL = ""
SOURCE_FILENAME = ""

if SOURCE_MODE == "drive":
    SOURCE_DRIVE_FILE_ID = input("ID do arquivo no Drive (.nup/.rar/.zip/.7z): ").strip()
elif SOURCE_MODE == "url":
    SOURCE_URL = getpass("URL HTTPS do seu dump (entrada oculta): ").strip()
    SOURCE_FILENAME = input("Nome do arquivo com extensão (ex.: jogo.rar ou jogo.nup): ").strip()
else:
    raise ValueError("Escolha 'drive' ou 'url'.")

GITHUB_PAT = getpass(
    "GitHub PAT com Contents: Read and write para KintTany (entrada oculta): "
).strip()
assert GITHUB_PAT, "PAT obrigatório para gravar o handoff na branch HookLink2iOS."


## 3. Preparar RomFS no Linux e iniciar Actions

O Colab gera `RomFS.img` diretamente do NUP (sem expandir todos os arquivos), faz upload do NUP, RomFS e overlay ao Drive, valida os hashes e atualiza somente `hooklink2ios/handoff.json` na branch `HookLink2iOS`. Isso é o único gatilho do workflow novo.


In [ ]:
RESULT = prepare(
    source_url=SOURCE_URL,
    source_filename=SOURCE_FILENAME,
    source_drive_id=SOURCE_DRIVE_FILE_ID,
    drive_parent_folder_id=DRIVE_PARENT_FOLDER_ID,
    drive=drive,
    github_token=GITHUB_PAT,
    repo_root=Path("/content/HookLink2iOS-repo"),
)

HANDOFF = RESULT["handoff"]
print("\nPreparação Linux concluída.")
print("Pasta do Drive:", HANDOFF["drive"]["run_folder_url"] or HANDOFF["drive"]["run_folder_id"])
print("NUP:", HANDOFF["nup"]["size"], "bytes")
print("RomFS:", HANDOFF["romfs"]["size"], "bytes")
print("RomFS identity:", HANDOFF["romfs"]["identity"])
print("Commit handoff:", RESULT["github"]["commit_sha"])
print("GitHub Actions:", RESULT["github"]["actions_url"])
